# 03 - Trigger a Copilot CLI review from a button click

Click the button below to POST to the Function App. The Function shallow-clones the sample repo, runs `copilot -p "..."` in BYOK mode against the Foundry deployment, and returns markdown that is rendered inline.

**Pre-requisites:** notebooks 01 and 02 completed; `env/.env` contains `AZURE_FUNCTION_URL` and `AZURE_FUNCTION_KEY`.

**Estimated time per click:** 30-90 seconds for the sample repo.

To make what's happening under the hood visible, the notebook now renders:
1. A **topology panel** with the concrete Azure resource names in the request path (button → Function App → Foundry deployment).
2. Direct **portal links** to the Function App and Application Insights component.
3. After each click, an **execution panel** with per-step timings, the exact prompt sent to Copilot, the provider config the CLI routed to, and a snapshot of the files Copilot actually saw after `git clone`.
4. A pre-filled **KQL snippet** for App Insights, filtered by the returned `invocation_id`, so you can jump straight to the traces for the specific call.

---

## Step 1 - Load environment

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

required = [
    "AZURE_SUBSCRIPTION_ID",
    "AZURE_RESOURCE_GROUP",
    "AZURE_FUNCTION_URL",
    "AZURE_FUNCTION_KEY",
    "AZURE_FUNCTION_APP_NAME",
    "AZURE_APP_INSIGHTS_NAME",
    "AZURE_AI_ENDPOINT",
    "AZURE_AI_DEPLOYMENT_NAME",
    "AZURE_CONTAINER_IMAGE_REFERENCE",
    "SAMPLE_REPO_URL",
]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing environment variables (rerun notebooks 01 and 02): {missing}")

subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]
resource_group = os.environ["AZURE_RESOURCE_GROUP"]
function_url = os.environ["AZURE_FUNCTION_URL"]
function_key = os.environ["AZURE_FUNCTION_KEY"]
function_app_name = os.environ["AZURE_FUNCTION_APP_NAME"]
app_insights_name = os.environ["AZURE_APP_INSIGHTS_NAME"]
ai_endpoint = os.environ["AZURE_AI_ENDPOINT"]
ai_deployment_name = os.environ["AZURE_AI_DEPLOYMENT_NAME"]
container_image_reference = os.environ["AZURE_CONTAINER_IMAGE_REFERENCE"]
sample_repo_url = os.environ["SAMPLE_REPO_URL"]

function_app_portal_url = (
    f"https://portal.azure.com/#@/resource/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}/providers/Microsoft.Web/sites/{function_app_name}/appServices"
)
app_insights_portal_url = (
    f"https://portal.azure.com/#@/resource/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}/providers/microsoft.insights/components/{app_insights_name}/overview"
)
app_insights_logs_url = (
    f"https://portal.azure.com/#@/resource/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}/providers/microsoft.insights/components/{app_insights_name}/logs"
)

print(f"Function URL   : {function_url}")
print(f"Function App   : {function_app_name}")
print(f"App Insights   : {app_insights_name}")
print(f"Foundry model  : {ai_deployment_name} @ {ai_endpoint}")
print(f"Target repo    : {sample_repo_url}")
print("Function key   : (loaded)" if function_key else "Function key   : MISSING")

## Step 2 - What happens when you click the button

Concrete request path per click, using the resource names from `env/.env`:

1. Notebook POSTs `{}` (plus `x-functions-key`) to `AZURE_FUNCTION_URL`.
2. The Function App runs the container image `AZURE_CONTAINER_IMAGE_REFERENCE` on an Elastic Premium (EP1) Linux plan.
3. Inside the container, the Python function shallow-clones `SAMPLE_REPO_URL` into a temp directory (`git clone --depth 1`).
4. It then runs `copilot -p "<review prompt>" --allow-all-tools --no-color` in that directory. The Copilot CLI is in BYOK mode — no GitHub login — because `COPILOT_PROVIDER_*` env vars point it at the Foundry model.
5. The CLI calls `AZURE_AI_ENDPOINT` (Foundry AIServices) targeting deployment `AZURE_AI_DEPLOYMENT_NAME`, using the Foundry account key as `COPILOT_PROVIDER_API_KEY`.
6. The CLI writes markdown to stdout; the function wraps it as JSON with timing, provider, and repo-snapshot metadata and returns it.

Run the next cell to render portal links for the deployed resources.

In [ ]:
from IPython.display import HTML, display

topology_html = f"""
<style>
  .demo-panel {{ font-family: -apple-system, Segoe UI, sans-serif; border: 1px solid #d0d7de;
                 border-radius: 6px; padding: 12px 16px; margin: 6px 0; background: #f6f8fa; }}
  .demo-panel h4 {{ margin: 0 0 8px 0; }}
  .demo-panel table {{ border-collapse: collapse; }}
  .demo-panel td {{ padding: 2px 8px 2px 0; vertical-align: top; font-size: 13px; }}
  .demo-panel td.k {{ color: #57606a; white-space: nowrap; }}
  .demo-panel td.v {{ font-family: Consolas, monospace; }}
  .demo-panel a {{ text-decoration: none; color: #0969da; }}
  .demo-panel a:hover {{ text-decoration: underline; }}
</style>
<div class="demo-panel">
  <h4>Deployed resources (from env/.env)</h4>
  <table>
    <tr><td class="k">Function App</td><td class="v">
      <a href="{function_app_portal_url}" target="_blank">{function_app_name}</a></td></tr>
    <tr><td class="k">Container image</td><td class="v">{container_image_reference}</td></tr>
    <tr><td class="k">Function URL</td><td class="v">{function_url}</td></tr>
    <tr><td class="k">App Insights</td><td class="v">
      <a href="{app_insights_portal_url}" target="_blank">{app_insights_name}</a>
      &nbsp;&middot;&nbsp;
      <a href="{app_insights_logs_url}" target="_blank">Open Logs (KQL)</a></td></tr>
    <tr><td class="k">Foundry endpoint</td><td class="v">{ai_endpoint}</td></tr>
    <tr><td class="k">Foundry deployment</td><td class="v">{ai_deployment_name}</td></tr>
    <tr><td class="k">Sample repo</td><td class="v">
      <a href="{sample_repo_url}" target="_blank">{sample_repo_url}</a></td></tr>
  </table>
</div>
"""
display(HTML(topology_html))

## Step 3 - Render the button

Click **Run Copilot Review** to trigger the function. Below the review markdown, you'll see:

- An **execution panel** with invocation id, per-step timings, provider echo (base URL + model, key redacted), and the top-N largest files Copilot saw after `git clone`.
- Collapsible **stdout / stderr tails** from the CLI (always shown, not just on error).
- A **KQL snippet** pre-filled with `operation_Id == "<invocation_id>"` and a UTC time window — paste it into the App Insights Logs blade to see the traces for that specific call.

The full response is also saved to `../outputs/last-review.json` for offline inspection.

In [ ]:
import html
import json
import time
from pathlib import Path

import ipywidgets as widgets
import requests
from IPython.display import HTML, Markdown, clear_output, display

outputs_dir = Path("../outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)
last_review_path = outputs_dir / "last-review.json"

button = widgets.Button(
    description="Run Copilot Review",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="260px"),
)
status = widgets.HTML(value="")
output_area = widgets.Output()


def _fmt_bytes(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"


def _render_execution_panel(payload: dict, client_wall_s: float) -> str:
    timing = payload.get("timing") or {}
    provider = payload.get("provider") or {}
    snap = payload.get("repo_snapshot") or {}
    top_files_rows = "".join(
        f"<tr><td class='v'>{html.escape(f['path'])}</td>"
        f"<td class='v' style='text-align:right'>{_fmt_bytes(f['bytes'])}</td></tr>"
        for f in (snap.get("top_files") or [])
    ) or "<tr><td colspan='2'><i>(no files)</i></td></tr>"

    return f"""
    <div class="demo-panel">
      <h4>How this was executed</h4>
      <table>
        <tr><td class="k">Invocation id</td><td class="v">{html.escape(payload.get('invocation_id',''))}</td></tr>
        <tr><td class="k">Started (UTC)</td><td class="v">{html.escape(payload.get('started_utc',''))}</td></tr>
        <tr><td class="k">Ended (UTC)</td><td class="v">{html.escape(payload.get('ended_utc',''))}</td></tr>
        <tr><td class="k">Timing</td><td class="v">
          clone {timing.get('clone_s','?')}s &middot;
          copilot {timing.get('copilot_s','?')}s &middot;
          total {timing.get('total_s','?')}s
          &nbsp;(client wall time {client_wall_s:.2f}s)</td></tr>
        <tr><td class="k">Provider</td><td class="v">
          type=<b>{html.escape(str(provider.get('type','')))}</b> &middot;
          model=<b>{html.escape(str(provider.get('model','')))}</b><br>
          base_url={html.escape(str(provider.get('base_url','')))}</td></tr>
        <tr><td class="k">Repo snapshot</td><td class="v">
          {snap.get('file_count','?')} files &middot; {_fmt_bytes(snap.get('total_bytes',0))} total</td></tr>
      </table>
      <details style="margin-top:8px">
        <summary><b>Prompt sent to Copilot</b></summary>
        <pre style="white-space:pre-wrap; background:#fff; padding:8px; border:1px solid #d0d7de; border-radius:4px">{html.escape(payload.get('prompt',''))}</pre>
      </details>
      <details style="margin-top:8px">
        <summary><b>Files Copilot saw (top by size)</b></summary>
        <table style="margin-top:6px">{top_files_rows}</table>
      </details>
    </div>
    """


def _render_kql_snippet(payload: dict) -> str:
    inv = payload.get("invocation_id") or ""
    started = payload.get("started_utc") or ""
    ended = payload.get("ended_utc") or ""
    kql = (
        f"union traces, requests, exceptions, dependencies\n"
        f"| where timestamp between (datetime({started}) .. datetime({ended}))\n"
        f'| where operation_Id == "{inv}" or customDimensions.InvocationId == "{inv}"\n'
        f"| project timestamp, itemType, message=coalesce(message, name), severityLevel, "
        f"resultCode, duration, customDimensions\n"
        f"| order by timestamp asc"
    )
    return f"""
    <div class="demo-panel">
      <h4>Trace this invocation in Application Insights</h4>
      <p style="margin:4px 0"><a href="{app_insights_logs_url}" target="_blank">Open the Logs blade</a>, then paste this KQL:</p>
      <pre style="white-space:pre-wrap; background:#fff; padding:8px; border:1px solid #d0d7de; border-radius:4px; font-size:12px">{html.escape(kql)}</pre>
    </div>
    """


def on_click(_):
    button.disabled = True
    status.value = f"<i>Requesting review of {sample_repo_url}\u2026 (typically 30-90s)</i>"
    with output_area:
        clear_output()
    start = time.monotonic()
    try:
        response = requests.post(
            function_url,
            headers={"x-functions-key": function_key},
            json={},
            timeout=220,
        )
    except requests.RequestException as ex:
        status.value = f"<span style='color:#b00'>Request failed: {ex}</span>"
        button.disabled = False
        return

    elapsed = round(time.monotonic() - start, 2)

    try:
        payload = response.json()
    except ValueError:
        payload = {"error": "Non-JSON response", "raw": response.text[:2000]}

    last_review_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    ok = response.status_code == 200 and "review" in payload
    total = (payload.get("timing") or {}).get("total_s", elapsed)
    if ok:
        status.value = (
            f"<span style='color:#080'>Review complete in {total}s "
            f"(client wall time {elapsed}s). Saved to {last_review_path}.</span>"
        )
    else:
        status.value = (
            f"<span style='color:#b00'>HTTP {response.status_code} in {elapsed}s. "
            f"See saved response at {last_review_path}.</span>"
        )

    with output_area:
        if ok:
            display(Markdown(f"### Copilot review of `{payload.get('repo', sample_repo_url)}`"))
            display(Markdown(payload["review"]))
        else:
            display(Markdown("### Error response"))
            display(Markdown("```json\n" + json.dumps(payload, indent=2) + "\n```"))

        display(HTML(_render_execution_panel(payload, elapsed)))

        if payload.get("stdout_tail"):
            display(Markdown(
                "<details><summary><b>stdout tail</b></summary>\n\n```\n"
                + payload["stdout_tail"] + "\n```\n</details>"
            ))
        if payload.get("stderr_tail"):
            display(Markdown(
                "<details><summary><b>stderr tail</b></summary>\n\n```\n"
                + payload["stderr_tail"] + "\n```\n</details>"
            ))

        if payload.get("invocation_id"):
            display(HTML(_render_kql_snippet(payload)))

    button.disabled = False


button.on_click(on_click)
display(widgets.VBox([widgets.HBox([button, status]), output_area]))

---

## Next

- Re-run the click as many times as you like; each invocation shallow-clones a fresh copy of the sample repo.
- Change the target repo by updating the `sampleRepoUrl` Bicep parameter and rerunning notebook 01, then setting the app setting: `az functionapp config appsettings set --name <app> --resource-group <rg> --settings SAMPLE_REPO_URL=<url>`.
- When finished, run **`X_destroy.ipynb`** to tear down all resources.